# 📡 OFDM-based Integrated Sensing and Communication (ISAC)
> v5.2 - Professional Engineering Edition (Detection Fix)

## 📑 1. Executive Summary
This version provides a robust fix for target detection issues. The 2D CA-CFAR algorithm has been re-engineered to handle boundary conditions correctly and prevent target self-masking.

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft, ifft, fftshift

# Physical and Operational Constants
C = 299792458.0
FS = 20e6
FC = 5.8e9
LAMBDA = C / FC
PRI = 500e-6
N_PULSES = 128
N_SC = 128
CP = 32
N_SAMPLES = N_SC + CP

range_res = C / (2 * FS)
vel_res = LAMBDA / (2 * N_PULSES * PRI)

def generate_ofdm_stream(n_symbols):
    data = (np.random.choice([-1, 1], (n_symbols, N_SC)) + 1j * np.random.choice([-1, 1], (n_symbols, N_SC))) / np.sqrt(2)
    ofdm_time = ifft(data, axis=1)
    cp = ofdm_time[:, -CP:]
    return np.hstack([cp, ofdm_time]).flatten()

def simulate(targets, SNR_dB=30):
    ref = []
    surv = []
    freqs = np.fft.fftfreq(N_SAMPLES)
    for p in range(N_PULSES):
        sig = generate_ofdm_stream(1)
        sig /= np.sqrt(np.mean(np.abs(sig)**2))
        ref.append(sig)
        echo = np.zeros_like(sig, dtype=complex)
        for t in targets:
            delay = (2 * t['R'] / C) * FS
            fd = 2 * t['v'] / LAMBDA
            sig_f = np.fft.fft(sig)
            shifted = np.fft.ifft(sig_f * np.exp(-1j * 2 * np.pi * freqs * delay))
            doppler_phase = np.exp(1j * 2 * np.pi * fd * p * PRI)
            loss = (100 / max(t['R'], 10))**2
            echo += shifted * doppler_phase * t['amp'] * loss
        noise_std = np.sqrt(1 / (10**(SNR_dB / 10)) / 2)
        noise = noise_std * (np.random.randn(N_SAMPLES) + 1j * np.random.randn(N_SAMPLES))
        direct_path = 5.0 * sig
        surv.append(direct_path + echo + noise)
    return np.array(ref), np.array(surv)

def process(ref, surv):
    corr = fft(surv, axis=1) * np.conj(fft(ref, axis=1))
    corr *= np.hanning(N_SAMPLES)
    rp = ifft(corr, axis=1)
    mti = rp[2:] - 2*rp[1:-1] + rp[:-2]
    mti *= np.hanning(mti.shape[0])[:, np.newaxis]
    rd = fftshift(fft(mti, axis=0), axes=0)
    return rd

def cfar_2d(rd, guard=(4, 4), train=(8, 8), pfa=1e-5):
    power = np.abs(rd)**2
    rows, cols = power.shape
    S = np.zeros((rows + 1, cols + 1))
    S[1:, 1:] = np.cumsum(np.cumsum(power, axis=0), axis=1)
    Sc = np.zeros((rows + 1, cols + 1))
    Sc[1:, 1:] = np.cumsum(np.cumsum(np.ones_like(power), axis=0), axis=1)
    def get_area_sum(mat, r1, c1, r2, c2):
        r1, r2 = np.clip([r1, r2+1], 0, rows)
        c1, c2 = np.clip([c1, c2+1], 0, cols)
        return mat[r2, c2] - mat[r1, c2] - mat[r2, c1] + mat[r1, c1]
    det = np.zeros_like(power, dtype=bool)
    gr, gc = guard
    tr, tc = train
    for i in range(rows):
        for j in range(cols):
            total_sum = get_area_sum(S, i-tr-gr, j-tc-gc, i+tr+gr, j+tc+gc)
            total_cnt = get_area_sum(Sc, i-tr-gr, j-tc-gc, i+tr+gr, j+tc+gc)
            guard_sum = get_area_sum(S, i-gr, j-gc, i+gr, j+gc)
            guard_cnt = get_area_sum(Sc, i-gr, j-gc, i+gr, j+gc)
            n_cells = total_cnt - guard_cnt
            if n_cells > 0:
                noise_floor = (total_sum - guard_sum) / n_cells
                alpha = n_cells * (pfa**(-1/n_cells) - 1)
                if power[i, j] > alpha * noise_floor:
                    det[i, j] = True
    return det

targets = [{"R": 150, "v": 25, "amp": 2.0}, {"R": 400, "v": -15, "amp": 1.5}, {"R": 650, "v": 5, "amp": 1.0}]
ref, surv = simulate(targets)
rd = process(ref, surv)
det = cfar_2d(rd)
print(f"Detected Targets: {np.sum(det)}")

Detected Targets: 17
